In [ ]:
import os, sys, numpy as np, torch, yaml
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm

# ================== 1. 定位项目根目录 ==================
# 方案A：直接指定（最稳定）
project_root = os.path.abspath(os.path.join(os.getcwd(), "."))
# 兼容从 src/ 或项目根运行：
if not os.path.exists(os.path.join(project_root, "configs", "config.yaml")):
    project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if not os.path.exists(os.path.join(project_root, "configs", "config.yaml")):
    project_root = os.path.abspath(os.path.join(os.path.dirname(os.getcwd()), ".."))

# 验证
if not os.path.exists(os.path.join(project_root, 'configs', 'config.yaml')):
    # 方案B：尝试从当前工作目录向上搜索
    print("硬编码路径未找到 config.yaml，尝试自动搜索...")
    search_from = os.getcwd()
    print(f"  当前工作目录: {search_from}")
    found = False
    for _ in range(5):
        if os.path.exists(os.path.join(search_from, 'configs', 'config.yaml')):
            project_root = search_from
            found = True
            break
        parent = os.path.dirname(search_from)
        if parent == search_from:
            break
        search_from = parent
    if not found:
        # 方案C：从笔记本所在目录搜索
        try:
            # 在 Jupyter 中通过 ipynb 的路径推断
            notebook_dir = os.path.dirname(os.path.abspath(''))
            search_from = notebook_dir
            for _ in range(5):
                if os.path.exists(os.path.join(search_from, 'configs', 'config.yaml')):
                    project_root = search_from
                    found = True
                    break
                parent = os.path.dirname(search_from)
                if parent == search_from:
                    break
                search_from = parent
        except:
            pass
    if not found:
        raise FileNotFoundError(
            "无法自动定位项目根目录。\n"
            "请修改上方 project_root = r'你的实际路径'"
        )

sys.path.insert(0, project_root)
os.chdir(project_root)
print(f"✅ 项目根目录: {project_root}")

# ================== 2. 加载配置 ==================
config_path = 'configs/config.yaml'
with open(config_path, 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
class_names = config['data']['class_names']
num_classes = config['data']['num_classes']

# ================== 3. 加载模型 ==================
from models.model_zoo.resnet1d import resnet18_1d, resnet34_1d

model_name = config['model']['name']
dropout = config['model'].get('dropout', 0.3)
if model_name == 'resnet34':
    model = resnet34_1d(in_channels=12, num_classes=num_classes, dropout=dropout)
else:
    model = resnet18_1d(in_channels=12, num_classes=num_classes, dropout=dropout)

checkpoint_path = config['paths']['checkpoints']
best_model_path = os.path.join(checkpoint_path, 'best_model.pth')
model.load_state_dict(torch.load(best_model_path, map_location=device))
model.to(device)
model.eval()
print(f"设备: {device} | 参数量: {sum(p.numel() for p in model.parameters()):,}")

# ================== 4. 决策阈值 ==================
# 从权威文件加载（优先级同 app/backend/utils.py 与 evaluate.py）：
#   1) configs/optimized_thresholds.yaml  2) reports/optimal_thresholds.json
import json as _json

def _load_thresholds(num_classes):
    cands = [
        os.path.join('configs', 'optimized_thresholds.yaml'),
        os.path.join('configs', 'optimized_thresholds.yml'),
        os.path.join('reports', 'optimal_thresholds.json'),
    ]
    for p in cands:
        if not os.path.exists(p):
            continue
        if p.endswith(('.yaml', '.yml')):
            with open(p, 'r', encoding='utf-8') as f:
                data = yaml.safe_load(f)
            if isinstance(data, dict):
                return [float(data[str(i)]) for i in range(num_classes)]
            return list(data)
        elif p.endswith('.json'):
            with open(p, 'r', encoding='utf-8') as f:
                data = _json.load(f)
            if 'thresholds' in data:
                return list(data['thresholds'])
    return [0.5] * num_classes

thresholds_list = _load_thresholds(num_classes)
print("决策阈值:", thresholds_list)

# ================== 5. 推理函数 ==================
def predict(model, signals, batch_size=128, device='cpu'):
    model.eval()
    dataset = TensorDataset(torch.as_tensor(signals, dtype=torch.float32))
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    all_probs = []
    with torch.no_grad():
        for (batch,) in tqdm(loader, desc='标准推理', unit='batch'):
            batch = batch.to(device)
            logits = model(batch)
            probs = torch.sigmoid(logits).cpu().numpy()
            all_probs.append(probs)
    probs = np.concatenate(all_probs, axis=0)
    preds = (probs >= np.array(thresholds_list)).astype(int)
    return preds, probs

# ================== 6. 推理 & 评估 ==================
data_dir = config['data']['processed_dir']
test_signals = np.load(os.path.join(data_dir, 'test', 'signals.npy')).astype(np.float32)
y_test = np.load(os.path.join(data_dir, 'labels', 'test_multilabel.npy')).astype(np.float32)

print(f"\n测试集大小: {test_signals.shape}")
preds, probs = predict(model, test_signals, batch_size=128, device=str(device))

np.savez('final_predictions.npz', preds=preds, probs=probs)
print("预测结果已保存至 final_predictions.npz")

from sklearn.metrics import f1_score, hamming_loss
macro_f1 = f1_score(y_test, preds, average='macro')
micro_f1 = f1_score(y_test, preds, average='micro')
hamming = hamming_loss(y_test, preds)
print(f"Hamming: {hamming:.4f}, Macro F1: {macro_f1:.4f}, Micro F1: {micro_f1:.4f}")
print("各类别 F1:")
for i, cls in enumerate(class_names):
    f1 = f1_score(y_test[:, i], preds[:, i])
    print(f"  {i}: {cls:6s} {f1:.4f}")

✅ 项目根目录: D:\Code (VS Code)\EGC_Model
设备: cpu | 参数量: 3,941,002

测试集大小: (2161, 12, 1000)


标准推理: 100%|██████████| 17/17 [00:52<00:00,  3.10s/batch]

预测结果已保存至 final_predictions.npz
Hamming: 0.1022, Macro F1: 0.6472, Micro F1: 0.7588
各类别 F1:
  0: SR     0.9079
  1: NORM   0.8355
  2: ABQRS  0.4211
  3: IMI    0.6033
  4: ASMI   0.6974
  5: LVH    0.4968
  6: NDT    0.4962
  7: LAFB   0.7188
  8: AFIB   0.7724
  9: ISC_   0.5227
